# Objectif 
L'objectif est d'analyser la propa d'une des émissions de la portion 4. 

Date de création : 24/11/2026

In [ ]:
import os
import sys
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy.io import loadmat
from matplotlib.lines import Line2D
from datetime import datetime, timedelta

In [ ]:
if os.name == "nt":  # Windows
    ROOT_PROGRAM = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd"
else:  # Linux
    ROOT_PROGRAM = "/home/program/ubf_tools"
sys.path.append(ROOT_PROGRAM)

from misc import equivalent_celerity
from publication.publication_figure import set_subfigures_abc_labels, color
from real_data_analysis.acouplane.src.acouplane_cst import *
from real_data_analysis.acouplane.src.acouplane_utils import *


In [ ]:
ROOT_IMG_PROPAGATION = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\acouplane\img\portion4\propagation"

# Chargement des données 

Les données sont stockées au format netcdf sous la forme d'un arbre (xarray DataTree) et regroupe l'ensemble des données associées à la portion d'intérêt. 

In [ ]:
portion_id = 4
fpath = f"{ACOUPLANE_PORTION_NC_ROOT_FPATH}_{portion_id}.nc"
portion = xr.open_datatree(fpath)
portion

# Rappel de la position de la portion 

In [ ]:
plot_portion_tir_pos(portion_tree=portion)

# Signaux d'intérêt 

Tracer des signaux étudiés. 

In [ ]:
c0 = 1508       # equivalent celerity 
for name, node in portion.position.children.items():
    ds_pos_section = node.to_dataset()
    # Propagation distance assuming strait line propagation path
    dpropa_direct = np.sqrt(ds_pos_section.src_rcv_geodesic_dist**2 + ds_pos_section.rcv_immersion**2)
    # Compute propagation time 
    tps_propagation_s = dpropa_direct / c0 
    # tps_propagation_s = ds_pos_section.src_rcv_geodesic_dist / c0 
    ds_pos_section["tps_propagation_s"] = tps_propagation_s
    ds_pos_section.tps_propagation_s.attrs = {
        "units": "s",
        "long_name": "Propagation time",
    }

    # Derive associated arrival dt 
    tps_propagation_ns = (tps_propagation_s * 1e9).round()
    tps_propagation_td = tps_propagation_ns.astype("timedelta64[ns]")
    ds_pos_section["tir_reception_datetime"] = tps_propagation_td + ds_pos_section.src_datetime

    # Update tree
    node.ds = ds_pos_section

In [ ]:
nb_tir = 1
offset1 = 200
offset2 = 1000
obs_ids = list(portion.pressure.children.items())[0][1].to_dataset().obs_id.values
# obs_ids = obs_ids[[0, 3, 5]]
for name, node in portion.pressure.children.items():
    ds_pressure_section = node.to_dataset()  
    # Associated ds_pos
    ds_pos_section = portion.position[name].to_dataset()
    tir_dt = ds_pos_section["tir_reception_datetime"].isel(src_datetime=slice(0, 0+nb_tir))

    fig, axs = plt.subplots(nrows=obs_ids.size, ncols=1, figsize=(16, 14), sharex=False)        
    axs = axs.flatten()
    for i, rcv_id in enumerate(obs_ids):
        tir_dt_rcv = tir_dt.sel(rcv_id=rcv_id)
        # Select associated data window 
        press_rcv = ds_pressure_section[f"pressure_{rcv_id}"].sel({f"time_{rcv_id}":slice(tir_dt_rcv.values[0]-np.timedelta64(offset1, 'ms'), tir_dt_rcv.values[-1]+np.timedelta64(offset2, 'ms'))})
        # Plot pressure signal 
        # Normalize 
        press_rcv = press_rcv / press_rcv.max()
        press_rcv.plot(label=rcv_id.upper(), color=color(i), ax=axs[i])
        # Plot arrivals time 
        axs[i].vlines(x=tir_dt_rcv, ymin=-1, ymax=1, color="k", linestyle="--")
        # Add legend
        axs[i].legend(loc="upper right")

    xlab = axs[0].get_xlabel()
    ylab = axs[0].get_ylabel()

    for ax in axs:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")

    fig.suptitle(f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}")
    fig.supxlabel(xlab)
    fig.supylabel(ylab)

    plt.show()

## Séléction d'une émission 


In [ ]:
nb_tir = 1
offset1 = 200
offset2 = 1000

# Select OBS 1, 2, 4, and 7
obs_ids = list(portion.pressure.children.items())[0][1].to_dataset().obs_id.values
# obs_ids = obs_ids[[0, 1, 2, 5]]
ds_pos_section = portion.position["sn1"].to_dataset().sel(rcv_id=obs_ids)
ds_pos_section = ds_pos_section.isel(src_datetime=slice(0, 0 + nb_tir))
ds_pressure_section = portion.pressure["sn1"].to_dataset()

tir_dt = ds_pos_section["tir_reception_datetime"]
fig, axs = plt.subplots(nrows=obs_ids.size, ncols=1, figsize=(16, 14), sharex=False)
axs = axs.flatten()
for i, rcv_id in enumerate(obs_ids):
    tir_dt_rcv = tir_dt.sel(rcv_id=rcv_id)
    # Select associated data window
    press_rcv = ds_pressure_section[f"pressure_{rcv_id}"].sel(
        {
            f"time_{rcv_id}": slice(
                tir_dt_rcv.values[0] - np.timedelta64(offset1, "ms"),
                tir_dt_rcv.values[-1] + np.timedelta64(offset2, "ms"),
            )
        }
    )
    # Plot pressure signal
    # Normalize
    press_rcv = press_rcv / press_rcv.max()
    press_rcv.plot(label=rcv_id.upper(), color=color(i), ax=axs[i])
    # Plot arrivals time
    axs[i].vlines(x=tir_dt_rcv, ymin=-1, ymax=1, color="k", linestyle="--")
    # Add legend
    axs[i].legend(loc="upper right")

xlab = axs[0].get_xlabel()
ylab = axs[0].get_ylabel()

for ax in axs:
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

fig.suptitle(
    f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}"
)
fig.supxlabel(xlab)
fig.supylabel(ylab)

# Bathymétrie dans la zone 

In [ ]:
ds_bathy = xr.open_dataset(ACOUPLANE_BATHY_NC_FPATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.colors import LightSource
from matplotlib.patches import Rectangle

# # =============================================================================
# # INPUT DATA
# # =============================================================================

# Extract data
z = ds_bathy["elevation"].values
lon = ds_bathy["lon"].values
lat = ds_bathy["lat"].values

# # Receiver dataset
# rx_lon = np.array([obs4_lon, obs7_lon])
# rx_lat = np.array([obs7_lat, obs7_lat])
# rx_id = [4, 7]

# Receiver dataset
rx_lon = ds_pos_section.rcv_lon.values
rx_lat = ds_pos_section.rcv_lat.values
# rx_id = [int(obs_id[-1]) for obs_id in ds_pos_section.rcv_id.values]
rx_id = ds_pos_section.rcv_id.values

# =============================================================================
# DEFINE ZOOM AREA FROM RECEIVER NETWORK
# =============================================================================

pad = 0.03  # degrees

zoom_extent = [
    rx_lon.min() - pad,
    rx_lon.max() + pad,
    rx_lat.min() - pad,
    rx_lat.max() + pad,
]

# =============================================================================
# HILLSHADE
# =============================================================================

ls = LightSource(
    azdeg=315,
    altdeg=45,
)

# Hillshade computed on bathymetry
rgb = ls.shade(
    -z,  # invert bathymetry for proper relief
    cmap=plt.cm.Blues,
    vert_exag=8,
    blend_mode="overlay",
)


# =============================================================================
# FIGURE
# =============================================================================

fig = plt.figure(figsize=(14, 10))

# -----------------------------------------------------------------------------
# MAIN MAP
# -----------------------------------------------------------------------------

ax = fig.add_axes(
    [0.05, 0.05, 0.90, 0.90],
    projection=ccrs.PlateCarree(),
)

ax.imshow(
    rgb,
    origin="lower",
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    transform=ccrs.PlateCarree(),
)

# Bathymetric contours
levels = np.arange(
    np.floor(z.min() / 100) * 100,
    0,
    1000,
)

cs = ax.contour(
    lon,
    lat,
    z,
    levels=levels,
    colors="k",
    linewidths=0.3,
    alpha=0.4,
    transform=ccrs.PlateCarree(),
)


# Zoom rectangle
rect = Rectangle(
    (zoom_extent[0], zoom_extent[2]),
    zoom_extent[1] - zoom_extent[0],
    zoom_extent[3] - zoom_extent[2],
    fill=False,
    edgecolor="red",
    linewidth=2,
    transform=ccrs.PlateCarree(),
    zorder=20,
)

ax.add_patch(rect)

# Grid
gl = ax.gridlines(
    draw_labels=True,
    linestyle=":",
    linewidth=0.5,
)

gl.top_labels = False
gl.right_labels = False

ax.set_title(
    # "Bathymetry and Receiver Network",
    "ACOUPLANE",
    fontsize=16,
)


# Create mask
land_mask = z >= 0

# Land color (RGB)
land_color = np.array([0.9, 0.9, 0.9])

# Replace land pixels by a uniform color
rgb_land = rgb.copy()
rgb_land[land_mask, :3] = land_color
rgb_land[~land_mask] = np.nan


ax.imshow(
    rgb_land,
    origin="lower",
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    transform=ccrs.PlateCarree(),
    zorder=16,
)

# Coastlines and land
ax.coastlines(resolution="10m", linewidth=1, zorder=17)
ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=17)

# =============================================================================
# INSET MAP
# =============================================================================

axins = fig.add_axes(
    [0.55, 0.10, 0.30, 0.30],  # left, bottom, width, height
    projection=ccrs.PlateCarree(),
)

# Bathymetry in inset
axins.imshow(
    rgb,
    origin="lower",
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    transform=ccrs.PlateCarree(),
)

# Bathymetric contours
levels = np.arange(
    np.floor(z.min() / 100) * 100,
    0,
    500,
)

# Contours
cs = axins.contour(
    lon,
    lat,
    z,
    levels=levels,
    colors="k",
    linewidths=0.3,
    alpha=1,
    transform=ccrs.PlateCarree(),
)


# Receivers
axins.scatter(
    rx_lon,
    rx_lat,
    s=80,
    c="crimson",
    marker="^",
    edgecolors="black",
    linewidths=0.8,
    transform=ccrs.PlateCarree(),
    zorder=40,
)

# Receiver labels
for rid, x, y in zip(rx_id, rx_lon, rx_lat):
    axins.annotate(
        str(rid),
        (x, y),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=7,
        bbox=dict(
            boxstyle="round,pad=0.2",
            fc="white",
            ec="none",
            alpha=0.7,
        ),
        zorder=41,
    )

# Zoom extent
axins.set_extent(zoom_extent)

# Grid
gl2 = axins.gridlines(
    draw_labels=True,
    linestyle=":",
    linewidth=0.3,
)

gl2.top_labels = False
gl2.right_labels = True
gl2.left_labels = False

axins.set_title(
    "Receiver Network",
    fontsize=10,
)


plt.show()

In [ ]:
# Plot source position for selected ranges
pad = 0.1  # degrees

zoom_extent = [
    rx_lon.min() - pad,
    rx_lon.max() + pad,
    rx_lat.min() - pad,
    rx_lat.max() + pad,
]

bathy_zoom = ds_bathy.sel(
    lon=slice(zoom_extent[0], zoom_extent[1]), lat=slice(zoom_extent[2], zoom_extent[3])
)


z = bathy_zoom["elevation"].values
lon = bathy_zoom["lon"].values
lat = bathy_zoom["lat"].values

# Hillshade
ls = LightSource(azdeg=315, altdeg=45)  # illumination azimuth  # illumination elevation
rgb = ls.shade(-z, cmap=plt.cm.Blues, vert_exag=5, blend_mode="overlay")

# Figure
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())


# Shaded bathymetry
ax.imshow(
    rgb,
    origin="lower",
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    transform=ccrs.PlateCarree(),
)

# Optional depth contours
levels = np.arange(np.floor(z.min() / 100) * 100, 0, 100)

cs = ax.contour(
    lon,
    lat,
    z,
    levels=levels,
    colors="k",
    linewidths=0.4,
    alpha=1,
    transform=ccrs.PlateCarree(),
)

ax.clabel(cs, fmt="%d m", fontsize=7)

# Coastlines and land
ax.coastlines(resolution="10m", linewidth=1, zorder=17)
ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=17)

# Grid
gl = ax.gridlines(draw_labels=True, linestyle=":")
gl.top_labels = False
gl.right_labels = False

ax.set_title("")

# Receivers
ax.scatter(
    rx_lon,
    rx_lat,
    s=80,
    c="crimson",
    marker="^",
    edgecolors="black",
    linewidths=0.8,
    transform=ccrs.PlateCarree(),
    zorder=10,
)

# Receiver labels
i = 0
sign_x = [1, -1, -1, 1, 1, 1, 1, -1]
sign_y = [-1, 1, 1, 1, 1, 1, -1, 1]
for rid, x, y in zip(rx_id, rx_lon, rx_lat):
    ax.annotate(
        str(rid).upper(),
        (x, y),
        xytext=(sign_x[i]*8, sign_y[i]*8),
        textcoords="offset points",
        fontsize=8,
        bbox=dict(
            boxstyle="round,pad=0.2",
            fc="red",
            ec="none",
            alpha=0.7,
        ),
        zorder=11,
    )
    i += 1

# Add source position
ax.scatter(
    ds_pos_section.tir_src_lon,
    ds_pos_section.tir_src_lat,
    s=100,
    c="limegreen",
    marker="*",
    edgecolors="black",
    linewidths=0.8,
    transform=ccrs.PlateCarree(),
    zorder=10,
    label="Source position"
)


# Zoom on the area of interest
ax.set_extent(zoom_extent)

ax.legend(fontsize=18, loc="lower left")

fpath = os.path.join(ROOT_IMG_PROPAGATION, "bathy_portion1.png")
fig.savefig(fpath)

# Modélisation de la propagation 

## Extraction des profils de bathymétrie 

In [ ]:
from get_data.bathymetry.bathy_profile_extraction import extract_bathy_profile

In [ ]:
# We need to rename elevation into bathymetry to use the functionbelow
xr_bathy = ds_bathy.rename(elevation="bathymetry")
xr_bathy

In [ ]:
bathy_profiles = {}
for rcv_id in ds_pos_section.rcv_id.values:
    r_profile, z_profile = extract_bathy_profile(
        xr_bathy=xr_bathy,
        start_lon=ds_pos_section.tir_src_lon.values[0],
        start_lat=ds_pos_section.tir_src_lat.values[0],
        stop_lon=ds_pos_section.sel(rcv_id=rcv_id).rcv_lon.values,
        stop_lat=ds_pos_section.sel(rcv_id=rcv_id).rcv_lat.values,
        range_resolution=10
    )
    z_profile = z_profile * -1  # Convert bathymetry to depth
    bathy_profiles[rcv_id] = {
            "range_m": r_profile,
            "depth_m": z_profile
    }

In [ ]:
# ax.scatter(
#     rx_lon,
#     rx_lat,
#     s=80,
#     c="crimson",
#     marker="^",
#     edgecolors="black",
#     linewidths=0.8,
#     transform=ccrs.PlateCarree(),
#     zorder=10,
# )


# # Add source position
# ax.scatter(
#     ds_pos_section.tir_src_lon,
#     ds_pos_section.tir_src_lat,
#     s=100,
#     c="limegreen",
#     marker="*",
#     edgecolors="black",
#     linewidths=0.8,
#     transform=ccrs.PlateCarree(),
#     zorder=10,
#     label="Source position",
# )

In [ ]:
z_source = 5
fig, axs = plt.subplots(
    nrows=ds_pos_section.sizes["rcv_id"], ncols=1, figsize=(16, 14), sharex=False
)
axs = axs.flatten()
for i, rcv_id in enumerate(ds_pos_section.rcv_id.values):
    axs[i].plot(
        bathy_profiles[rcv_id]["range_m"],
        bathy_profiles[rcv_id]["depth_m"],
        label=rcv_id.upper(),
        color=color(i),
    )

    # Add source
    axs[i].scatter(
        bathy_profiles[rcv_id]["range_m"][0],
        z_source,
        s=100,
        c="limegreen",
        marker="*",
        edgecolors="black",
        linewidths=0.8,
        zorder=10,
        # label="Source",
    )
    # Add receiver
    axs[i].scatter(
        bathy_profiles[rcv_id]["range_m"][-1],
        bathy_profiles[rcv_id]["depth_m"][-1] - 1,
        s=100,
        c="crimson",
        marker="^",
        edgecolors="black",
        linewidths=0.8,
        # label="Receiver",
        # zorder=10,
    )

    # Add legend
    axs[i].legend(loc="upper right", fontsize=12)

for ax in axs:
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    ax.invert_yaxis()

legend_handles = [
    Line2D(
        [0],
        [0],
        linestyle="None",
        marker="*",
        markersize=14,
        markerfacecolor="limegreen",
        markeredgecolor="black",
        markeredgewidth=0.8,
        label="Source",
    ),
    Line2D(
        [0],
        [0],
        linestyle="None",
        marker="^",
        markersize=10,
        markerfacecolor="crimson",
        markeredgecolor="black",
        markeredgewidth=0.8,
        label="Receiver",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="outside right center",
    ncols=1,
    fontsize=20,
)

fig.suptitle(
    f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}"
)
fig.supxlabel("Range [m]")
fig.supylabel("Depth [m]")

set_subfigures_abc_labels(axs, x_pos=0.5, y_pos=1.02, fontsize=20, ha="center", va="bottom")

fpath = os.path.join(ROOT_IMG_PROPAGATION, "bathy_profiles_portion1.png")
fig.savefig(fpath)

## Profil de célérité 

In [ ]:
celerity = xr.open_datatree(ACOUPLANE_CELERITY_NC_FPATH)
celerity

In [ ]:
# selected_profile = "T7_00003"
# selected_profile = "T7_00019"
selected_profile = "T5_00008"

# selected_profile = "T0_00020"

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
for name, node in celerity.children.items():
    if name == selected_profile:
        ds_celerity = node.to_dataset()
        ds_celerity.celerity.plot(y="depth", ax=ax, yincrease=False, label=name)

        # Apply rolling mean
        smooth_celerity_profile = ds_celerity.celerity.rolling(
            depth=100, center=True, min_periods=1
        ).mean()
        smooth_celerity_profile.plot(y="depth", ax=ax, yincrease=False, label=f"{name} (smoothed)", color="k")

        # Use smooth profile
        z_ssp, cp_ssp = smooth_celerity_profile.depth.values, smooth_celerity_profile.values
        # Subsample profile
        z_ssp = z_ssp[::20]
        cp_ssp = cp_ssp[::20]
        ax.plot(cp_ssp, z_ssp, "o", color="k", markersize=4, label=f"{name} (subsampled)")

ax.legend(loc="lower right", ncols=2, fontsize=10)
# plt.ylim(20, 0)
fpath = os.path.join(ROOT_IMG_PROPAGATION, "celerity_portion1.png")
fig.savefig(fpath)

## Calcul de la propa avec Kraken 

In [ ]:
bathy_profiles

In [ ]:
# Use smooth profile 
z_ssp, cp_ssp = smooth_celerity_profile.depth.values, smooth_celerity_profile.values
# Subsample profile 
z_ssp = z_ssp[::20]
cp_ssp = cp_ssp[::20]

In [ ]:
from real_data_analysis.acouplane.src.analyse_portion.run_receivers_propagation import run_all_receivers

In [ ]:
root_simu = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\acouplane\src\analyse_portion\portion4\propagation"

In [ ]:
run_all_receivers(
    bathy_profiles,
    z_ssp,
    cp_ssp,
    run_kraken=False,
    root=root_simu,
)  # d'abord : fichiers + figures d'environnement

In [ ]:
results, failed = run_all_receivers(
    bathy_profiles,
    z_ssp,
    cp_ssp,
    run_kraken=True,
    test_mode=True,
    root=root_simu,
)  # puis un test
# run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True)

In [25]:
results, failed = run_all_receivers(
    bathy_profiles,
    z_ssp,
    cp_ssp,
    run_kraken=True,
    only=ds_pos_section.rcv_id.values,
    root=root_simu,
)
# # run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True, only=["obs1"])
# # run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True, only=["obs2"])
# # run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True, only=["obs4"])
# # run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True, only=["obs7"])
# run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True, only=["obs5"])
# run_all_receivers(bathy_profiles, z_ssp, cp_ssp, run_kraken=True, only=["obs6"])

KeyboardInterrupt: 

## Lecture des résultats et calcul de la réponse impulsionnelle

In [ ]:
results

In [ ]:
import pickle
from pathlib import Path

# root_simulation = Path(
#     r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis"
#     r"\acouplane\src\analyse_portion\portion1\propagation"
# )
root_simulation = Path(root_simu)


def load_receiver(name, root=root_simulation, mmap=True):
    """Return (pressure_field, field_pos, freq) saved by run_receiver for one receiver."""
    d = Path(root) / str(name)
    # mmap_mode="r": the file is read from disk on demand (useful for the ~6 GB arrays)
    pressure_field = np.load(d / "pressure_field.npy", mmap_mode="r" if mmap else None)
    freq = np.load(d / "freq.npy")
    with open(d / "field_pos.pkl", "rb") as fh:
        field_pos = pickle.load(fh)
    return pressure_field, field_pos, freq


# # One receiver
# pressure_field, field_pos, freq = load_receiver("obs1")
# print(pressure_field.shape, pressure_field.dtype)  # (n_freq, n_z, n_r)

# All the receivers (lazy: nothing is loaded in RAM until you slice the arrays)
# receivers = ds_pos_section.rcv_id.values
results = {name: load_receiver(name) for name in list(results.keys())}

In [ ]:
# results["obs1"][2]

In [ ]:
results.keys()

In [ ]:
# ---- Frequency / time grids (computed once, outside the loop)
from real_data_analysis.acouplane.src.analyse_portion.run_receivers_propagation import (
    NT,
    FS,
)

print(FS)
freq = np.fft.rfftfreq(n=NT, d=1.0 / FS)
Sf = np.ones_like(freq)  # source spectrum on the FULL rfft grid (includes f = 0)

nfft = 2 * (freq.size - 1)  # = nt
df = freq[1] - freq[0]
nfft_inv = 4 * nfft  # Jensen et al. (2000) p.616: dt < 1 / (8 fmax)
dt = 1.0 / (nfft_inv * df)
time_vector = np.arange(nfft_inv) * dt  # exactly nfft_inv samples

k0 = 2 * np.pi * freq / c0
norm_factor = np.exp(1j * k0) / (4 * np.pi)

tau = ds_pos_section["tps_propagation_s"]
# rcv_ids = ds_pos_section.rcv_id.values
rcv_ids = list(results.keys())
impulse_responses = {}
# fig, axs = plt.subplots(nrows=len(rcv_ids), ncols=1, figsize=(16, 14), squeeze=False)
# axs = axs.flatten()

# tau_rcvs = [0.742, 0.681, 0.75, 0.332]
tau_rcvs = [0.406, 0.49, 0, 0.20, 0, 0, 0]

for i, rcv_id in enumerate(rcv_ids):
    # tau_rcv = tau.sel(rcv_id=rcv_id).values[0]
    tau_rcv = tau_rcvs[i]

    # Transfer function at the receiver position (results[rcv_id] = (pressure, field_pos, freq_sim))
    pressure_rcv, pos_rcv, f_sim = results[rcv_id]
    pressure_rcv = pressure_rcv.squeeze()  # (n_freq_sim, n_z, n_r)
    r_grid, z_grid = pos_rcv["r"]["r"], pos_rcv["r"]["z"]

    idx_r = np.argmin(np.abs(r_grid - bathy_profiles[rcv_id]["range_m"][-1]))
    zrcv = bathy_profiles[rcv_id]["depth_m"][-1] - 1  # 1 m above the seafloor
    idx_z = np.argmin(np.abs(z_grid - zrcv))
    tf_sim = np.asarray(pressure_rcv[:, idx_z, idx_r], dtype=complex)

    # Put the computed frequencies on the full rfft grid (f = 0 and skipped bins -> 0)
    tf = np.zeros(freq.size, dtype=complex)
    tf[np.rint(f_sim / df).astype(int)] = tf_sim

    # Propagation delay, Y = S H
    delay_rcv = np.exp(-1j * 2 * np.pi * tau_rcv * freq)
    y_f = Sf * tf * norm_factor * delay_rcv

    # Inverse to get time response
    y_t = np.fft.irfft(y_f, n=nfft_inv)

    impulse_responses[rcv_id] = y_t
    # axs[i].plot(time_vector, y_t)
    # axs[i].set_title(str(rcv_id))
    # axs[i].set_ylabel("Amplitude")

# axs[-1].set_xlabel("Time (s)")
# fig.tight_layout()

In [ ]:
import scipy.signal as sp 

In [ ]:
tir_dt = ds_pos_section["tir_reception_datetime"]
for i, rcv_id in enumerate(rcv_ids):
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(16, 8), sharex=False)

    tir_dt_rcv = tir_dt.sel(rcv_id=rcv_id)
    # Select associated data window
    press_rcv = ds_pressure_section[f"pressure_{rcv_id}"].sel(
        {
            f"time_{rcv_id}": slice(
                tir_dt_rcv.values[0] - np.timedelta64(offset1, "ms"),
                tir_dt_rcv.values[-1] + np.timedelta64(offset2, "ms"),
            )
        }
    )
    # Plot kraken ir
    y_t = impulse_responses[rcv_id]
    y_t = y_t / y_t.max()

    sos = sp.butter(
        N=10,
        Wn=10,
        btype="highpass",
        fs=FS,
        output="sos",
    )
    y_t_filtered = sp.sosfilt(sos, y_t)

    ax.plot(
        time_vector,
        y_t_filtered,
        color=color(0),
        linestyle="-",
        label=f"{rcv_id.upper()} Kraken",
        linewidth=3,
    )
    # Plot pressure signal
    # Normalize
    press_rcv = press_rcv / press_rcv.max()
    press_rcv = press_rcv - np.mean(press_rcv)
    # Time vector
    t_ns = press_rcv[f"time_{rcv_id}"] - press_rcv[f"time_{rcv_id}"].values[0]
    t_s = t_ns.values / np.timedelta64(1, "s")
    # Plot
    sig = press_rcv.values
    sos = sp.butter(
        N=10,
        Wn=[10, freq.max()],
        btype="bandpass",
        fs=1 / (t_s[1] - t_s[0]),
        output="sos",
    )
    filtered = sp.sosfilt(sos, sig)
    filtered = filtered / np.max(filtered)
    ax.plot(t_s, filtered, label=rcv_id.upper(), color="k", linewidth=1)
    # ax.plot(t_s, press_rcv.values, label=rcv_id.upper(), color="k", linewidth=1)

    # Add legend
    ax.legend(loc="upper right")

    # xlab = axs[0].get_xlabel()
    # ylab = axs[0].get_ylabel()

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

    fig.suptitle(
        f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}"
    )
    # fig.supxlabel(xlab)
    # fig.supylabel(ylab)
    # plt.xlim(0.1, 0.8)

    fpath = os.path.join(
        ROOT_IMG_PROPAGATION, f"signal_vs_kraken_ir_{rcv_id.upper()}.png"
    )
    fig.savefig(fpath)